In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Gumawa ng pass manager para sa dynamical decoupling

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda namin ang paggamit ng mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Ipinapakita ng pahinang ito kung paano gamitin ang [`PadDynamicalDecoupling`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling) pass para magdagdag ng teknik sa pagpigil ng error na tinatawag na _dynamical decoupling_ sa Circuit.

Gumagana ang dynamical decoupling sa pamamagitan ng pagdaragdag ng mga pulse sequence (kilala bilang _dynamical decoupling sequences_) sa mga idle na Qubit para paikutin sila sa paligid ng Bloch sphere, na nagkakansela ng epekto ng mga noise channel at nagpipigil ng decoherence. Ang mga pulse sequence na ito ay katulad ng mga refocusing pulse na ginagamit sa nuclear magnetic resonance. Para sa kumpletong paglalarawan, tingnan ang [A Quantum Engineer's Guide to Superconducting Qubits](https://arxiv.org/abs/1904.06560).

Dahil ang `PadDynamicalDecoupling` pass ay gumagana lamang sa mga naka-schedule na Circuit at gumagamit ng mga Gate na hindi necessarily basis gates ng ating target, kakailanganin mo rin ang mga pass na [`ALAPScheduleAnalysis`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.ALAPScheduleAnalysis) at [`BasisTranslator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes.BasisTranslator).

Ginagamit ng halimbawang ito ang `ibm_fez`, na na-initialize na noon. Kunin ang impormasyon ng `target` mula sa `backend` at i-save ang mga operation name bilang `basis_gates` dahil kailangang baguhin ang `target` para magdagdag ng timing information para sa mga Gate na ginagamit sa dynamical decoupling.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend("ibm_fez")

target = backend.target
basis_gates = list(target.operation_names)

Gumawa ng `efficient_su2` Circuit bilang halimbawa. Una, i-transpile ang Circuit sa Backend dahil kailangang idagdag ang mga dynamical decoupling pulse pagkatapos ma-transpile at ma-schedule ang Circuit. Kadalasan, mas epektibo ang dynamical decoupling kapag maraming idle time sa mga quantum Circuit — ibig sabihin, may mga Qubit na hindi ginagamit habang aktibo ang iba. Ganito ang kaso sa Circuit na ito dahil ang mga two-qubit `ecr` Gate ay inilapat nang sunud-sunod sa ansatz na ito.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)
pm = generate_preset_pass_manager(1, target=target, seed_transpiler=12345)
qc_t = pm.run(qc)
qc_t.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/8228f889-806a-4873-b1da-27c9795d5f5c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/8228f889-806a-4873-b1da-27c9795d5f5c-0.svg)

Ang isang dynamical decoupling sequence ay isang serye ng mga Gate na nagkokompose sa identity at pantay-pantay ang pagitan sa oras. Halimbawa, magsimula sa paggawa ng simpleng sequence na tinatawag na XY4 na binubuo ng apat na Gate.

In [3]:
from qiskit.circuit.library import XGate, YGate

X = XGate()
Y = YGate()

dd_sequence = [X, Y, X, Y]

Dahil sa regular na timing ng mga dynamical decoupling sequence, kailangang idagdag ang impormasyon tungkol sa `YGate` sa `target` dahil ito ay *hindi* isang basis gate, samantalang ang `XGate` ay basis gate. Alam natin *a priori* na ang `YGate` ay may parehong duration at error tulad ng `XGate`, kaya makukuha lang natin ang mga katangiang iyon mula sa `target` at idagdag ang mga ito para sa mga `YGate` na object. Ito rin ang dahilan kung bakit nai-save nang hiwalay ang `basis_gates`, dahil idinaragdag natin ang `YGate` na instruksyon sa `target` kahit na hindi ito aktwal na basis gate ng `ibm_fez`.

In [4]:
from qiskit.transpiler import InstructionProperties

y_gate_properties = {}
for qubit in range(target.num_qubits):
    y_gate_properties.update(
        {
            (qubit,): InstructionProperties(
                duration=target["x"][(qubit,)].duration,
                error=target["x"][(qubit,)].error,
            )
        }
    )

target.add_instruction(YGate(), y_gate_properties)

Ang mga ansatz Circuit tulad ng `efficient_su2` ay may mga parameter, kaya kailangang bigyan ng halaga ang mga ito bago ipadala sa Backend. Dito, mag-assign ng mga random na parameter.

In [5]:
import numpy as np

rng = np.random.default_rng(1234)
qc_t.assign_parameters(
    rng.uniform(-np.pi, np.pi, qc_t.num_parameters), inplace=True
)

Susunod, i-execute ang mga custom na pass. I-instantiate ang `PassManager` gamit ang `ALAPScheduleAnalysis` at `PadDynamicalDecoupling`. Patakbuhin muna ang `ALAPScheduleAnalysis` para magdagdag ng timing information tungkol sa quantum Circuit bago idagdag ang mga regularly-spaced na dynamical decoupling sequence. Ang mga pass na ito ay pinapatakbo sa Circuit gamit ang `.run()`.

In [6]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

dd_pm = PassManager(
    [
        ALAPScheduleAnalysis(target=target),
        PadDynamicalDecoupling(target=target, dd_sequence=dd_sequence),
    ]
)
qc_dd = dd_pm.run(qc_t)

Gamitin ang visualization tool na [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) para makita ang timing ng Circuit at kumpirmahin na ang mga regularly-spaced na `XGate` object at `YGate` object ay lumilitaw sa Circuit.

In [7]:
from qiskit.visualization import timeline_drawer

timeline_drawer(qc_dd, idle_wires=False, target=target)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/cb73e2c4-ab05-4f15-91ae-2fab64028d6e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/cb73e2c4-ab05-4f15-91ae-2fab64028d6e-0.svg)

Sa huli, dahil ang `YGate` ay hindi aktwal na basis gate ng ating Backend, manu-manong i-apply ang `BasisTranslator` pass (ito ay default na pass, ngunit ito ay naipapatakbo bago ang scheduling, kaya kailangang i-apply uli). Ang session equivalence library ay isang library ng mga circuit equivalence na nagbibigay-daan sa Transpiler na i-decompose ang mga Circuit sa mga basis gate, gaya rin ng tinukoy bilang argumento.

In [8]:
from qiskit.circuit.equivalence_library import (
    SessionEquivalenceLibrary as sel,
)
from qiskit.transpiler.passes import BasisTranslator

qc_dd = BasisTranslator(sel, basis_gates)(qc_dd)
qc_dd.draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/aaa27ee4-1965-41bf-abd2-1d9176af6dc4-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/dynamical-decoupling-pass-manager/extracted-outputs/aaa27ee4-1965-41bf-abd2-1d9176af6dc4-0.svg)

Ngayon, wala nang `YGate` na object sa ating Circuit, at mayroon nang malinaw na timing information sa anyo ng mga `Delay` gate. Ang na-transpile na Circuit na ito na may dynamical decoupling ay handa nang ipadala sa Backend.

## Mga susunod na hakbang

> **Tip:** - Para matutunan kung paano gamitin ang function na `generate_preset_passmanager` sa halip na sumulat ng sariling mga pass, magsimula sa paksa na [Transpilation default settings and configuration options](defaults-and-configuration-options).
>     - Subukan ang gabay na [Compare transpiler settings](/guides/circuit-transpilation-settings#compare-transpiler-settings).
>     - Tingnan ang [Transpile API documentation.](https://docs.quantum-computing.ibm.com/api/qiskit/transpiler)